# 03. 시공간 분석 (Time & Space Analysis)

## 목표
- 시간대 × 요일별 이용 패턴 분석
- 대여소별 우천 감소율 계산 및 시각화
- 지도 기반 공간 분석

## ULTRA-THINK Framework: T - Transform (변환하기)

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.utils.data_loader import load_station_master
from src.utils.preprocessing import calculate_rain_impact, merge_station_master
from src.utils.visualization import (
    plot_heatmap_hour_weekday,
    plot_station_rain_impact,
    create_rain_impact_map
)

import warnings
warnings.filterwarnings('ignore')

## 1. 데이터 로딩

In [ ]:
# 전처리된 데이터 로딩
df = pd.read_csv("../data/processed/merged_hourly_202509.csv", encoding="utf-8")
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])

# 대여소 마스터 로딩
station = load_station_master(file_path="../data/external/서울시 따릉이대여소 마스터 정보.csv")

print(f"데이터 로딩 완료: {len(df):,}개 레코드")

## 2. 시간대 × 요일별 이용 패턴

In [ ]:
# 히트맵 생성
plot_heatmap_hour_weekday(df, save_path="../outputs/figures/heatmap_hour_weekday.png")

## 3. 대여소별 우천 감소율 분석

In [ ]:
# 우천 감소율 계산
rain_impact = calculate_rain_impact(df)

In [ ]:
# TOP 10 / LOW 10 대여소
print("\n=== 우천 감소율 TOP 10 (가장 많이 감소) ===")
print(rain_impact.head(10))

print("\n=== 우천 감소율 LOW 10 (가장 적게 감소 또는 증가) ===")
print(rain_impact.tail(10))

In [ ]:
# 시각화
plot_station_rain_impact(rain_impact, top_n=10, save_path="../outputs/figures/bar_station_rain_impact_toplow.png")

In [ ]:
# 우천 감소율 저장
rain_impact.to_csv("../data/processed/rain_impact_by_station.csv", encoding="utf-8")
print("✅ 우천 감소율 데이터 저장 완료")

## 4. 지도 기반 시각화

In [ ]:
# 지도 생성
create_rain_impact_map(
    station_df=station,
    rain_impact_series=rain_impact,
    save_path="../outputs/figures/map_rain_impact.html"
)

In [ ]:
# 지도 미리보기 (Jupyter에서)
from IPython.display import IFrame
IFrame(src='../outputs/figures/map_rain_impact.html', width=900, height=600)

## 5. 시간대별 우천 민감도 분석

In [ ]:
# 시간대별 맑은 날 vs 비 오는 날
clear_hourly = df[df['비여부'] == 0].groupby('기준_시간')['전체_건수'].mean()
rainy_hourly = df[df['비여부'] == 1].groupby('기준_시간')['전체_건수'].mean()

plt.figure(figsize=(14, 6))
plt.plot(clear_hourly.index, clear_hourly.values, marker='o', label='맑은 날', linewidth=2)
plt.plot(rainy_hourly.index, rainy_hourly.values, marker='s', label='비 오는 날', linewidth=2)
plt.title('시간대별 맑은 날 vs 비 오는 날 이용량', fontsize=14, fontweight='bold')
plt.xlabel('시간대 (시)', fontsize=12)
plt.ylabel('평균 이용 건수', fontsize=12)
plt.legend(loc='upper left', fontsize=11)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/hourly_clear_vs_rainy.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. 다음 단계

- ✅ 시공간 분석 완료
- ✅ 대여소별 우천 감소율 계산 완료
- 다음: `04_case_study_heavyrain.ipynb`에서 집중호우 사례 분석